In [ ]:
#GROUP DEFINITION

"""This notebook used to classify impaired cognition using the previously established threshold. 
It first codifies whether a test is 'impaired', then merges the prospective memory data - 
which was not subject to normative modelling, but does give data about imapriement in memory. 
It then merges dementia data pulled from the RAP that includes the date of diagnosis of all-
cause dementia. Cognitively healthy (0 impaired tests) and cognitive impaired (≥ 2 impaired tests)
are then classified."""

import pandas as pd
import numpy as np
import os

data_dir = "data_dir"
case_dir = "Caselists" #The tabular data containing the ID and the date of diagnosis of all cause dementia
out_dir  = "output_dir/"

os.makedirs(out_dir, exist_ok=True)

In [ ]:
#CODIFY IMPAIRMENT
"""
Loads cleaned cognitive test data and assigns a binary impairment flag at 1.5 SD.
Reverse-scored tests (higher raw = worse) use z_score > threshold; all others use z_score < -threshold.
The threshold sweep (1.0 / 1.5 / 2.0 / 2.5) is run in 05f.
"""

REVERSE_TESTS = ["PMat", "RT", "TMA", "TMB"]

df = pd.read_csv(f"{data_dir}/cv_results_all_tests_cleaned.csv")

def is_impaired(row, threshold=1.5):
    if row["test_name"] in REVERSE_TESTS:
        return 1 if row["z_score"] > threshold else 0
    return 1 if row["z_score"] < -threshold else 0

df["impaired"] = df.apply(is_impaired, axis=1)
df.to_csv(f"{data_dir}/cv_results_all_tests_coded.csv", index=False)

print(f"Rows: {len(df):,}  |  Impaired: {df['impaired'].sum():,} ({100*df['impaired'].mean():.2f}%)")
print(df.groupby("test_name")["impaired"].mean().mul(100).round(2).rename("% impaired").to_string())

In [ ]:
#MERGE PMEM
"""
Appends the separately scored PMem (prospective memory) test in long format.
PMem impairment is pre-calculated in 05a_PMem_transform.r (binary; no z-score available).
"""

pmem = pd.read_csv(f"{data_dir}/PMem_long.csv")
df = pd.concat([df, pmem], axis=0, ignore_index=True)
df = df.sort_values(["eid", "age_at_test", "test_name", "instance"], na_position="last")

df.to_csv(f"{data_dir}/cv_results_all_tests_merged.csv", index=False)

print(f"Merged: {len(df):,} rows  |  {df['eid'].nunique():,} participants  |  {df['test_name'].nunique()} tests")
print(df["test_name"].value_counts().sort_index())

In [ ]:
#MERGE DEMENTIA
"""
Loads diagnosis dates and date of birth.
Computes:
  - age_at_dementia_diagnosis (per participant)
  - time_to_dementia_diagnosis (per test instance, in years; negative = test AFTER diagnosis)
The time-to-diagnosis column supports Cox proportional hazards survival modelling.
"""

month_map = {
    "January":1, "February":2, "March":3, "April":4, "May":5, "June":6,
    "July":7, "August":8, "September":9, "October":10, "November":11, "December":12,
}

dem = pd.read_csv(f"{case_dir}/ID_dateDem.txt", sep="\t", header=None,
                  names=["eid", "date_of_dx"])
dem["date_of_dx"] = pd.to_datetime(dem["date_of_dx"], format="%d/%m/%Y")
dem["diag_decimal"] = (dem["date_of_dx"].dt.year
                       + (dem["date_of_dx"].dt.month - 1) / 12
                       + dem["date_of_dx"].dt.day / 365.25)

dob = pd.read_csv(f"{case_dir}/id_DOB.csv") #DOB data for participants with dementia - exported from RAP
dob["month_num"] = dob["month"].map(month_map)
dob["birth_decimal"] = dob["year"] + (dob["month_num"] - 0.5) / 12

dem = dem.merge(dob[["eid", "birth_decimal"]], on="eid", how="left")
dem["age_at_dx"] = (dem["diag_decimal"] - dem["birth_decimal"]).round(1)

df = df.merge(dem[["eid", "age_at_dx"]], on="eid", how="left")

# Per-instance time-to-event: years between test and diagnosis (positive = test BEFORE diagnosis)
df["time_to_dx"] = df["age_at_dx"] - df["age_at_test"]

# Construct proper survival data for Cox / landmark analyses
# Cases: time = test to diagnosis ; controls: time = test to end of follow-up

end_of_followup_decimal = 2025 + 8/12   # Sept 2025 

# Everyone gets an age at end of follow-up
df = df.merge(dob[["eid", "birth_decimal"]], on="eid", how="left")
df["age_at_followup_end"] = end_of_followup_decimal - df["birth_decimal"]

# time_to_event: years from test to diagnosis (cases) or end of follow-up (controls)
df["time_to_event"] = np.where(
    df["age_at_dx"].notna(),
    df["age_at_dx"]           - df["age_at_test"],   # cases
    df["age_at_followup_end"] - df["age_at_test"]    # controls
)

# Event indicator: 1 only if dementia diagnosed AFTER this test (incident)
df["event"] = ((df["age_at_dx"].notna()) & (df["time_to_event"] > 0)).astype(int)

# Flag prevalent cases (test done after diagnosis) for exclusion in Cox/landmark
df["prevalent"] = ((df["age_at_dx"].notna()) & (df["time_to_event"] <= 0)).astype(int)

print(f"\nSurvival data constructed:")
print(f"  Incident events: {df['event'].sum():,} obs  |  "
      f"{df[df['event']==1]['eid'].nunique():,} participants")
print(f"  Censored:        {((df['event']==0) & (df['prevalent']==0)).sum():,} obs")
print(f"  Prevalent:       {df['prevalent'].sum():,} obs (test after diagnosis - exclude)")


print("Age at dementia diagnosis:")
print(dem["age_at_dx"].describe().round(1).to_string())
print(f"\nTime to diagnosis (per instance, dementia patients only):")
print(df.loc[df['age_at_dx'].notna(), 'time_to_dx'].describe().round(2).to_string())
print(f"\nObservations with diagnosis age merged: {df['age_at_dx'].notna().sum():,}")

In [ ]:
#TAG DEMENTIA
"""
Flags all test instances for participants who eventually receive a dementia diagnosis,
regardless of whether the test occurred before or after that diagnosis.
"""

dementia_ids = set(dem["eid"].unique())
df["dem_all"] = df["eid"].isin(dementia_ids).astype(int)

print(f"dem_all = 1:  {df['dem_all'].sum():,} observations  |  {df[df['dem_all']==1]['eid'].nunique():,} participants")

In [ ]:
#DEFINE CONTROLS
"""
Control group (HC): participants with NO impaired test results across any instance.
Derived directly from the data — no external case list required.
All test instances for HC participants are saved to ./hc/ for downstream analyses.
"""

os.makedirs("hc", exist_ok=True)

impaired_eids = set(df.loc[df["impaired"] == 1, "eid"])
hc_eids      = set(df["eid"]) - impaired_eids

df["hc"] = df["eid"].isin(hc_eids).astype(int)

hc_df = df[df["hc"] == 1].sort_values(["eid", "test_name", "instance"])
hc_df.to_csv("hc/impaired_0_all_tests.csv", index=False)

n_total_p = df["eid"].nunique()
n_hc     = len(hc_eids)
pct_hc   = 100 * n_hc / n_total_p

print(f"HC (0 impaired):    {n_hc:,} participants ({pct_hc:.2f}%)")
print(f"HC test instances:  {len(hc_df):,}")
print(f"Saved → hc/impaired_0_all_tests.csv")

In [ ]:
#DEFINE IC
"""
IC at 1.5 SD, same-instance criterion: >=2 impaired tests in at least one assessment instance.
age_at_ic is the age at the first qualifying instance (earliest instance meeting the criterion).
All test instances for IC participants are saved to ./IC/.
"""

os.makedirs("IC", exist_ok=True)

# Count impaired tests per participant per instance (uses existing 1.5 SD impaired column)
inst_counts = (df.groupby(["eid", "instance"])["impaired"].sum()
                 .reset_index()
                 .rename(columns={"impaired": "n_impaired"}))

# First instance where >=2 tests impaired
first_IC = (inst_counts[inst_counts["n_impaired"] >= 2]
               .sort_values("instance")
               .groupby("eid").first()
               .reset_index()[["eid", "instance"]])

# Age at that instance (one age value per eid+instance)
first_IC_age = (df.merge(first_IC, on=["eid", "instance"])
                   .drop_duplicates(subset=["eid"])
                   [["eid", "age_at_test"]]
                   .rename(columns={"age_at_test": "age_at_ic"}))

df = df.merge(first_IC_age, on="eid", how="left")
IC_eids = set(first_IC["eid"])
df["ic"] = df["eid"].isin(IC_eids).astype(int)

IC_df = df[df["ic"] == 1].sort_values(["eid", "instance", "test_name"])
IC_df.to_csv("IC/IC_1p5sd_same_instance.csv", index=False)

n_IC     = len(IC_eids)
n_total_p = df["eid"].nunique()

print(f"IC (1.5 SD, same-instance):  {n_IC:,} participants ({100*n_IC/n_total_p:.2f}%)")
print(f"IC test instances:            {len(IC_df):,}")
print(f"Mean age at IC:               {first_IC_age['age_at_ic'].mean():.1f} years")
print(f"Saved → IC/IC_1p5sd_same_instance.csv")

In [ ]:
#SAVE STATE
"""
Saves the fully annotated dataset with group flags and time-to-event for downstream analyses
(05f threshold sweep, 05d descriptives, Cox PH survival modelling).
"""

out_path = f"{out_dir}/cv_results_all_tests_merged_with_groups.csv"
df.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nGroup flags:")
for col in ["dem_all", "hc", "ic"]:
    n_obs = int(df[col].sum())
    n_par = df[df[col]==1]["eid"].nunique()
    print(f"  {col}: {n_obs:,} obs  |  {n_par:,} participants")

In [ ]:
#DEFINE BORDERLINE (Bord)
"""
Borderline group: participants who are impaired on exactly ONE test within an instance,
and never reach the IC criterion (>=2 impaired in the same instance) at any instance.
Mutually exclusive with both HC (0 impaired) and IC (>=2 same-instance) by construction.
age_at_bord is the age at the first qualifying instance.
All test instances for Bord participants are saved to ./bord/.
"""

os.makedirs("bord", exist_ok=True)
df = pd.read_csv(f"{data_dir}/cv_results_all_tests_merged_with_groups.csv")

# Impaired count per participant per instance (recomputed so this cell can run standalone)
inst_counts_b = (df.groupby(["eid", "instance"])["impaired"].sum()
                   .reset_index()
                   .rename(columns={"impaired": "n_impaired"}))

# Participants whose maximum same-instance impaired count is exactly 1
max_per_eid = inst_counts_b.groupby("eid")["n_impaired"].max()
bord_eids   = set(max_per_eid[max_per_eid == 1].index)

# First instance with exactly 1 impaired test
first_bord = (inst_counts_b[(inst_counts_b["eid"].isin(bord_eids)) &
                            (inst_counts_b["n_impaired"] == 1)]
                .sort_values("instance")
                .groupby("eid").first()
                .reset_index()[["eid", "instance"]])

first_bord_age = (df.merge(first_bord, on=["eid", "instance"])
                    .drop_duplicates(subset=["eid"])
                    [["eid", "age_at_test"]]
                    .rename(columns={"age_at_test": "age_at_bord"}))

df = df.merge(first_bord_age, on="eid", how="left")
df["bord"] = df["eid"].isin(bord_eids).astype(int)

bord_df = df[df["bord"] == 1].sort_values(["eid", "instance", "test_name"])
bord_df.to_csv("bord/BORD_1p5sd_same_instance.csv", index=False)

n_bord    = len(bord_eids)
n_total_p = df["eid"].nunique()

# Stricter alternative (exactly 1 impaired test in total, across all instances) - reported only
total_per_eid = df.groupby("eid")["impaired"].sum()
n_bord_total  = int((total_per_eid == 1).sum())

print(f"Bord (exactly 1 impaired, same-instance):  {n_bord:,} participants ({100*n_bord/n_total_p:.2f}%)")
print(f"Bord test instances:                       {len(bord_df):,}")
print(f"Mean age at Bord:                          {first_bord_age['age_at_bord'].mean():.1f} years")
print(f"Saved -> bord/BORD_1p5sd_same_instance.csv")
print(f"\n[check] Mutually exclusive - overlap with HC: {int((df['bord'] & df['hc']).sum())} obs  |  "
      f"overlap with IC: {int((df['bord'] & df['ic']).sum())} obs")
print(f"[alt]   Exactly 1 impaired test in total (all instances): {n_bord_total:,} participants")
